In [18]:
import pandas as pd

vcf_file_path = '../carcinoma_normal_sniffles_filter/sniffles/Panc1.vcf'
vcf_df = pd.read_csv(vcf_file_path, sep='\t', comment='#', header=None)

vcf_df.columns = ['CHROM', 'POS', 'ID', 'REF', 'ALT', 'QUAL', 'FILTER', 'INFO', 'FORMAT', 'SAMPLE']

# Extract unique SVTYPE values from the INFO column
#unique_svtypes = vcf_df['INFO'].str.extract(r'SVTYPE=([^;]+)')[0].unique()
# Save unique SV types to a file
#pd.Series(unique_svtypes).to_csv('SVTYPES/PD51_CTS_P25.csv', index=False, header=['SVTYPE'])

In [19]:
vcf_df

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,SAMPLE
0,chr10,382861,Sniffles2.DEL.6EDS0,ttgtgGAGGCCTAGACTGgtgtgtgcgcgtgtgtgtgtgttgtgga...,N,56,PASS,PRECISE;SVTYPE=DEL;SVLEN=-62;END=382923;SUPPOR...,GT:GQ:DR:DV,1/1:25:0:9
1,chr10,383204,Sniffles2.INS.CS0,N,CTGATGTGTGTGTGTGTTGTGGAGGCCTAGGCTAGATGTGTGTGTG...,57,PASS,PRECISE;SVTYPE=INS;SVLEN=98;END=383204;SUPPORT...,GT:GQ:DR:DV,1/1:22:0:8
2,chr10,413861,Sniffles2.DEL.6F2S0,CCCGGGAAGCCCGCGGTTGCTCTCACTGCCGCACAGCCCCAGGTGC...,N,56,PASS,PRECISE;SVTYPE=DEL;SVLEN=-233;END=414094;SUPPO...,GT:GQ:DR:DV,1/1:30:0:11
3,chr10,463524,Sniffles2.INS.FS0,N,CACCTGTCAGAGCTCGGACAGGCCTCCCCGCTCCACCTGCACCTGT...,52,PASS,PRECISE;SVTYPE=INS;SVLEN=264;END=463524;SUPPOR...,GT:GQ:DR:DV,0/1:11:2:3
4,chr10,578482,Sniffles2.DEL.6FES0,TGTGAGAGAGTATGGGTgagagagagagtatgggtgtgagagagag...,N,55,PASS,PRECISE;SVTYPE=DEL;SVLEN=-70;END=578552;SUPPOR...,GT:GQ:DR:DV,1/1:33:0:12
...,...,...,...,...,...,...,...,...,...,...
14124,chrY,59001011,Sniffles2.BND.23AS53,N,N]chr21:11054314],34,PASS,"PRECISE;SVTYPE=BND;SUPPORT=11;COVERAGE=30,30,4...",GT:GQ:DR:DV,0/1:31:28:11
14125,chrY,59002944,Sniffles2.INS.74S53,N,TATTAAAATTTCAATCAACAGTCACAAAAGCAGACTAATAAAGCAG...,32,PASS,PRECISE;SVTYPE=INS;SVLEN=60;END=59002944;SUPPO...,GT:GQ:DR:DV,1/1:60:0:44
14126,chrY,59004521,Sniffles2.DEL.105S53,aaggaaagaaaagaaagaaaagaaagaaagaaaagaaagaaagaaa...,N,28,PASS,PRECISE;SVTYPE=DEL;SVLEN=-172;END=59004693;SUP...,GT:GQ:DR:DV,1/1:60:0:38
14127,chrY,59013760,Sniffles2.INS.75S53,N,GTAGCTGGGATTACAGGCACCTGCCGCCACACCTAGCAAATTTTTG...,26,PASS,PRECISE;SVTYPE=INS;SVLEN=133;END=59013760;SUPP...,GT:GQ:DR:DV,1/1:60:0:27


In [20]:
# Define a list of standard chromosomes
standard_chromosomes = ['chr' + str(i) for i in range(1, 23)] + ['chrX', 'chrY']

# Extract information from the INFO column
def parse_info(info_str):
    info_dict = {}
    for item in info_str.split(';'):
        if '=' in item:
            key, value = item.split('=')
            info_dict[key] = value
        else:
            info_dict[item] = True
    return info_dict

# Initialize lists to store the extracted information
chrom = []
start = []
stop = []
size = []
svtype = []
strand = []

# Extract relevant information from the VCF data
for index, row in vcf_df.iterrows():
    info = parse_info(row['INFO'])
    svtype_value = info.get('SVTYPE')
    if svtype_value not in ['TRA', 'BND'] and row['CHROM'] in standard_chromosomes:
        chrom.append(row['CHROM'])
        start.append(row['POS'])
        stop.append(info.get('END'))
        size.append(info.get('SVLEN'))
        svtype.append(svtype_value)
        strand.append(info.get('STRAND'))

# Create a DataFrame with the extracted information
filtered_sv_df = pd.DataFrame({
    'CHROM': chrom,
    'START': start,
    'STOP': stop,
    'SIZE': size,
    'SVTYPE': svtype,
    'STRAND': strand
})

In [21]:
filtered_sv_df

,CHROM,START,STOP,SIZE,SVTYPE,STRAND
0,chr10,382861,382923,-62,DEL,+-
1,chr10,383204,383204,98,INS,+-
2,chr10,413861,414094,-233,DEL,+-
3,chr10,463524,463524,264,INS,+-
4,chr10,578482,578552,-70,DEL,+-
...,...,...,...,...,...,...
13711,chrY,58997469,58997469,70,INS,+-
13712,chrY,59002944,59002944,60,INS,+-
13713,chrY,59004521,59004693,-172,DEL,+-
13714,chrY,59013760,59013760,133,INS,+-


In [22]:
# Check for None or NaN values
none_or_nan = filtered_sv_df.isna().sum()

# Check for zero values
zero_values = (filtered_sv_df == 0).sum()

# Combine the results into a DataFrame
combined_check = pd.DataFrame({
    'None or NaN': none_or_nan,
    'Zero values': zero_values
})

print("Combined check for None, NaN, and zero values in DataFrame:")
print(combined_check)

Combined check for None, NaN, and zero values in DataFrame:
        None or NaN  Zero values
CHROM             0            0
START             0            0
STOP              0            0
SIZE              0            0
SVTYPE            0            0
STRAND            0            0


In [23]:
filtered_sv_df.to_csv('../carcinoma_normal_sniffles_filter/simple_csv/Panc1.csv', index=False)